In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from analisis_reporte_telemetria import (
    cargar_datos_gestion_bus,
    limpieza_numerica,
    procesar_fechas_y_llaves,
    join_unique,
    extraer_caracter
)

# Movimiento Intercompañia

In [2]:
# Uso
path2 = Path(r"..\data\gestion-bus")
    
df_combustible_movimientos = cargar_datos_gestion_bus(path2)

Cantidad de Registros: 124095


In [ ]:
# Clave para analizar Entregas / Remito
df_combustible_movimientos = procesar_fechas_y_llaves(df_combustible_movimientos, 'Fecha', 'Almacen', 'Fecha Remito')
df_combustible_movimientos_ = df_combustible_movimientos.copy()

## Seleccion Almacen para Analizar
df_combustible_lujan = df_combustible_movimientos_[(df_combustible_movimientos_['Almacen']== "903 BARRACAS Lujan")]

# Clave para analizar Recargas
df_combustible_lujan = procesar_fechas_y_llaves(df_combustible_lujan, 'Fecha', 'Almacen', 'Fecha')


In [7]:
df_combustible_lujan['Tipo'].value_counts()

Tipo
Carga de Combustible              13700
Entrega de Combustible              394
Trasvase Combustible (Egreso)        16
Trasvase Combustible (Ingreso)       13
Name: count, dtype: int64

## Movimientos y ajustes

In [ ]:
tipos_movimientos = ['Trasvase Combustible (Egreso)',
'Trasvase Combustible (Ingreso)',
'Corrección Negativa',
'Corrección Positiva'
]

df_movimientos = df_combustible_lujan[df_combustible_lujan['Tipo'].isin(tipos_movimientos)]

df_movimientos_agrupado = df_movimientos.groupby(
    ['Año_mes',  'unico_mes','unico_mes_Almacen','Almacen', 'Tipo', 'Tanque']
).agg(
    Litros_Entregados_Sum=('Litros', 'sum'),
    Cantidad_veces=('Litros', 'count'),
    Observacion = ( 'Observación', join_unique),
    Usuario=('Usuario', join_unique),
    Fecha_Hora = ('Fecha Hora', join_unique)

).reset_index()

df_movimientos_agrupado.head(2)
df_movimientos_agrupado.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Kardex\0_Trasvase-Tanques.xlsx", index=False)

,Año_mes,unico_mes,unico_mes_Almacen,Almacen,Tipo,Tanque,Litros_Entregados_Sum,Cantidad_veces,Observacion,Usuario,Fecha_Hora
0,2025-11,903 BARRACAS Lujan_2025-11,903 BARRACAS Lujan_2025-11,903 BARRACAS Lujan,Trasvase Combustible (Egreso),001 Tanque 1,-15000.0,2,,mlromero,11/11/2025 08:09
1,2025-11,903 BARRACAS Lujan_2025-11,903 BARRACAS Lujan_2025-11,903 BARRACAS Lujan,Trasvase Combustible (Egreso),002 Tanque 2,-13800.0,1,,mlromero,29/11/2025 22:35


# Recargas y Entregas

In [14]:
df_entrega = df_combustible_lujan[df_combustible_lujan['Tipo']=="Entrega de Combustible"]
df_recargas = df_combustible_lujan[df_combustible_lujan['Tipo']=="Carga de Combustible"]


<a id="seccion_clave_unica"></a>
## 4. Recargas Gestion Bus

In [ ]:
df_recargas = extraer_caracter(df_recargas, 'Tanque')
df_recargas = procesar_fechas_y_llaves(df_recargas, 'Fecha', 'Tanque', 'Fecha')
df_recargas['Date_fx'] = pd.to_datetime(df_recargas['Fecha'], format='%d/%m/%Y', errors='coerce').dt.date


In [12]:

# Por ahora no vamos a ver como se distribuye por tanque ese va ser otro paso 
df_combustible_recargas_agrupado = df_recargas.groupby(
    ['Año_mes',  'unico_mes', 'Tipo','Date_fx', 'Almacen', 'Tanque']
).agg(
    Litros_GB_Recargas=('Litros', 'sum'),
    Cantidad_veces=('Litros', 'count'),
    Surtidor = ( 'Surtidor', join_unique), 
    # Usuario=('Usuario', join_unique),

).reset_index()
#df_combustible_agrupado_entregas['Date_fx'] = pd.to_datetime(df_combustible_agrupado_entregas['Date_fx'])

df_combustible_recargas_agrupado['unico_almacen'] = df_combustible_recargas_agrupado['Almacen'] + '_' + df_combustible_recargas_agrupado['Tanque'].astype(str) + '_' + df_combustible_recargas_agrupado['Date_fx'].astype(str) + '_' + df_combustible_recargas_agrupado['Litros_GB_Recargas'].astype(str)


df_combustible_recargas_agrupado.head(2)
#df_combustible_recargas_agrupado.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Kardex\2-Agrupacion-recargas_GB.xlsx", index=False)
#df_recargas.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Kardex\2-Recargas-Completo-GB.xlsx", index=False)

,Año_mes,unico_mes,Tipo,Date_fx,Almacen,Tanque,Litros_GB_Recargas,Cantidad_veces,Surtidor,unico_almacen
0,2025-11,1_2025-11,Carga de Combustible,2025-11-03,903 BARRACAS Lujan,1,-635.0,3,006 Surtidor 6,903 BARRACAS Lujan_1_2025-11-03_-635.0
1,2025-11,1_2025-11,Carga de Combustible,2025-11-04,903 BARRACAS Lujan,1,-15247.0,52,006 Surtidor 6,903 BARRACAS Lujan_1_2025-11-04_-15247.0


# Entregas

In [ ]:

# Por ahora no vamos a ver como se distribuye por tanque ese va ser otro paso 
df_combustible_lujan = df_combustible_lujan.groupby(
    ['Año_mes',  'unico_mes', 'unico_semana_almacen' ,'unico_mes_Almacen', 'Sociedad','Tipo', 'NroPedido','Almacen', 'Date_fx' ]
).agg(
    Litros_GB=('Litros', 'sum'),
    Cantidad_veces=('Litros', 'count'),
   # Date_fx = ( 'Fecha Remito', join_unique),
    Remito = ('Nro. Remito', join_unique),
    L_Entregados = ('Litros Entregado', 'max'),
    L_Solicitados = ('LitrosSolicitado', 'max'),
    Nro_Presinto=( 'Nro. Precinto', join_unique),
   
    Usuario=('Usuario', join_unique),

).reset_index()



df_combustible_lujan['unico_almacen'] = df_combustible_lujan['Almacen'] + '_' + df_combustible_lujan['Date_fx'].astype(str) + '_' + df_combustible_lujan['Litros_GB'].astype(str)


df_combustible_lujan['Date_fx']  = pd.to_datetime(df_combustible_lujan['Date_fx'],  dayfirst=True, errors='coerce').dt.normalize().astype('datetime64[ns]')

df_combustible_lujan.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\Agrupacion-Combustible-por-Almacen_GB.xlsx", index=False)

## Analisis Recargas por Almacen

In [ ]:

def limpieza_fecha(df, columna_fecha='Date_fx'):
    # 1. Copia de seguridad
    df = df.copy()
    
    # 2. Definimos nulos comunes para pre-limpieza
    common_nulls = ["-", " ", "N/A", "nd", "n/d", "None", "", "nan", "NaN", "unnamed"]
    
    if columna_fecha in df.columns:
        # 3. Limpieza básica de strings y reemplazo de nulos conocidos
        df[columna_fecha] = df[columna_fecha].astype(str).str.strip().replace(common_nulls, np.nan)
        
        # 4. Conversión a Datetime
        # errors='coerce' transformará cualquier texto que no sea fecha en NaT (Not a Time)
        df[columna_fecha] = pd.to_datetime(df[columna_fecha], dayfirst=True, errors='coerce')
        
        # 5. Manejo de nulos (Incluir "0" o una fecha mínima)
        # Nota: Las columnas de fecha en Pandas no aceptan el número 0 directamente si quieres 
        # mantener el tipo datetime. Aquí tienes dos opciones:
        
        # Opción A: Convertir a texto y poner "0" (si solo lo quieres para reporte)
        # df[columna_fecha] = df[columna_fecha].dt.strftime('%Y-%m-%d').fillna("0")
        
        # Opción B: Mantener como fecha y poner una fecha muy antigua (ej. 1900-01-01)
        # # Esto es mejor para poder seguir filtrando por fechas luego.
        # fecha_defecto = pd.Timestamp('1900-01-01')
        # df[columna_fecha] = df[columna_fecha].fillna(fecha_defecto)
        
    return df

In [ ]:
#df_combustible_lujan = limpieza_fecha(df_combustible_lujan) 

In [ ]:

df_combustible_lujan['unico_almacen'] = df_combustible_lujan['Almacen'] + '_' + df_combustible_lujan['Date_fx'].astype(str)+ '_' + df_combustible_lujan['Litros_GB'].astype(str)


df_combustible_lujan["Dif GB vs Entreg"] = df_combustible_lujan['Litros_GB']-df_combustible_lujan['L_Entregados']

df_combustible_lujan.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\Agrupacion-Combustible-por-Almacen_GB.xlsx", index=False)
#df_combustible_lujan.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\GB-Analisis-pedido_lujan_v2.xlsx", index=False)


# anterior

In [ ]:
# Llenar NaNs para poder restar sin errores
conciliacion_completa['Litros_Telemetria'] = conciliacion_completa['Litros_Telemetria'].fillna(0)
conciliacion_completa['Litros_GB'] = conciliacion_completa['Litros_GB'].fillna(0)

# Calcular diferencia individual
conciliacion_completa['Diferencia_Litros'] = conciliacion_completa['Litros_Telemetria'] - conciliacion_completa['Litros_GB']

# 1. Diferencia acumulada POR ALMACÉN (Correcto)
conciliacion_completa['Diferencia_Acumulada'] = conciliacion_completa.groupby('Fecha_Inicio')['Diferencia_Litros'].cumsum()

# 2. Suma móvil POR ALMACÉN (Correcto)
conciliacion_completa['Suma_Compensatoria'] = (
    conciliacion_completa.groupby('NroPedido')['Diferencia_Litros'] # Nro. Remito   Date_fx
    .rolling(window=2)
    .sum()
    .reset_index(level=0, drop=True)
)


def clasificar_error(row):
    if abs(row['Diferencia_Litros']) <= 50: # Tolerancia técnica normal
        return 'OK - Conciliado'
    elif abs(row['Suma_Compensatoria']) <= 50:
        return 'Compensado (Desfase de tiempo)'
    elif row['Diferencia_Litros'] > 300:
        return 'Sobrante (Posible carga no registrada)'
    elif row['Diferencia_Litros'] < -300:
        return 'Faltante (Posible merma o robo)'
    else:
        return 'Revisar manualmente'

conciliacion_completa['Estado_Critico'] = conciliacion_completa.apply(clasificar_error, axis=1)

conciliacion_completa.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\Conciliacion\prueba_remito_V2.xlsx", index=False)


In [ ]:
def separar_y_convertir_fechas(df, columna_origen):
    # 1. Limpiamos espacios en blanco al inicio/final y aseguramos que sea string
    temp_col = df[columna_origen].astype(str).str.strip()

    # 2. Dividimos la columna en dos partes por el primer espacio
    # n=1 asegura que solo divida una vez (separa fecha de lo demás)
    split_data = temp_col.str.split(' ', n=1, expand=True)
    
    # Asignamos a columnas temporales (col 0 es fecha, col 1 es hora)
    df['raw_date'] = split_data[0]
    df['raw_time'] = split_data[1]

    # 3. Convertimos la Fecha (Mes/Día/Año)
    # Usamos format='%m/%d/%Y' para ser estrictos con tu formato

    df['Date_hora'] = pd.to_datetime(df['raw_date'], format='%d/%m/%Y %H:%M', errors='coerce').dt.date
    df['Date_fx_reg'] = pd.to_datetime(df['raw_date'], format='%d/%m/%Y', errors='coerce').dt.date

    # 4. Convertimos la Hora
    # Si la fila no tenía hora, el split dio None, lo manejamos con errors='coerce'
    # .dt.time extrae solo la hora del objeto datetime resultante
    df['Time_fx_reg'] = pd.to_datetime(df['raw_time'], errors='coerce').dt.time

    # 5. Limpieza de columnas auxiliares
    df = df.drop(columns=['raw_date', 'raw_time'])
    
    return df

lujan = separar_y_convertir_fechas(lujan, 'Fecha Hora')

lujan.dtypes
# 3. Convertir a Datetime y NORMALIZAR (mantiene el tipo datetime64)
    # Quitamos .dt.date y ponemos .dt.normalize()
lujan['Date_fx_reg'] = pd.to_datetime(lujan['Date_fx_reg'], format='%d/%m/%Y', errors='coerce').dt.normalize()

# 4. Convertir la Hora (esta sí queda como object/time, lo cual es normal)
lujan['Time_fx_reg'] = pd.to_datetime(lujan['Time_fx_reg'], errors='coerce').dt.time

In [ ]:
filtro_lujan = lujan[(lujan['Date_fx'] >= '2025-10-01')& (lujan['Date_fx'] <= '2026-04-18')]


In [ ]:
lujan_almacen= lujan.groupby(['Date_fx','Año_mes', 'unico_almacen']).agg({ 
    'Litros': 'sum',
    'NroPedido': 'count',
    'LitrosSolicitado':'last'
}).reset_index()  
len(lujan_almacen)

In [ ]:
lujan_almacen['Litros'].sum()
print(f"registro LUJAN GB {len(lujan_almacen['Litros'])},suma {lujan_almacen['Litros'].sum()}") 

print(f"registro LUJAN telemetria {len(dfs_agrupado_telemetria['Ingresos'])},suma {dfs_agrupado_telemetria['Ingresos'].sum()}")

In [ ]:
lujan_almacen['Date_fx'] = pd.to_datetime(lujan_almacen['Date_fx'], format='%d/%m/%Y', errors='coerce')
dfs_agrupado_telemetria['Date_fx'] = pd.to_datetime(dfs_agrupado_telemetria['Date_fx'], format='%d/%m/%Y', errors='coerce')

In [ ]:

# 1. Asegurar formato fecha y ORDENAR (obligatorio para merge_asof)
dfs_agrupado_telemetria = dfs_agrupado_telemetria.sort_values("Date_fx")
lujan_almacen = lujan_almacen.sort_values("Date_fx")

# 2. Ejecutar el merge_asof
# Solo usamos 'unico_almacen' en el by para que coincida el lugar.
# La diferencia de litros la manejaremos después.
prueba_2 = pd.merge_asof(
    dfs_agrupado_telemetria,
    lujan_almacen,
    on="Date_fx",
    by='unico_almacen',  # Solo agrupamos por el ID del tanque/almacén
    tolerance=pd.Timedelta(days=2), # Tolerancia de 1 día (ajústalo si el desfase es mayor)
    direction='nearest'
)

In [ ]:
# 3. Calcular la diferencia entre lo que dice la telemetría (Ingresos) 
# y lo que dice el remito/almacén (Litros)
prueba_2['Diferencia_Litros'] = prueba_2['Ingresos'] - prueba_2['Litros']

# Asegúrate de que los datos sigan ordenados por fecha
prueba_2 = prueba_2.sort_values(['unico_almacen', 'Date_fx'])

# 1. Calculamos la diferencia acumulada por almacén
# Esto nos dirá si el error se arrastra o se compensa con el tiempo
prueba_2['Diferencia_Acumulada'] = prueba_2.groupby('Date_fx')['Diferencia_Litros'].cumsum()

# 2. Detectar compensaciones automáticas (Suma móvil de 2 o 3 registros)
# Si la suma de la diferencia actual + la anterior es cercana a 0, hubo una compensación
prueba_2['Suma_Compensatoria'] = prueba_2.groupby('Date_fx')['Diferencia_Litros'].rolling(window=2).sum().reset_index(0, drop=True)

# v2

In [ ]:
# 1. Ejecutar el merge_asof original
prueba_2 = pd.merge_asof(
    dfs_agrupado_telemetria.sort_values("Date_fx"),
    lujan_almacen.sort_values("Date_fx"),
    on="Date_fx",
    by='unico_almacen',
    tolerance=pd.Timedelta(days=2),
    direction='nearest'
)

# 2. Encontrar qué registros de lujan_almacen NO están en prueba_2
# Usamos un indicador para ver qué falta
almacen_en_prueba = prueba_2[['Date_fx', 'unico_almacen', 'Litros']].dropna()

# Identificamos los registros de lujan_almacen que no cruzaron
perdidos = lujan_almacen[~lujan_almacen.set_index(['Date_fx', 'unico_almacen']).index.isin(
    prueba_2.set_index(['Date_fx', 'unico_almacen']).index
)].copy()

# 3. Unir los registros perdidos al dataset de conciliación
# Llenamos 'Ingresos' con 0 porque son cargas de almacén que la telemetría no vio
perdidos['Ingresos'] = 0
conciliacion_completa = pd.concat([prueba_2, perdidos], ignore_index=True).sort_values(['unico_almacen', 'Date_fx'])


# Filtro dias revision

In [ ]:


# telemetria_filtrada = telemetria[telemetria['Date_fx'].isin(fechas_a_revisar)].copy()

# print(f"Registros encontrados: {len(telemetria_filtrada)}")
# telemetria_filtrada.head(2)

In [ ]:

# 1. Convertimos ambas columnas a datetime y eliminamos la hora (se vuelve 00:00:00)
#telemetria['Date_fx'] = pd.to_datetime(telemetria['Date_fx']).dt.normalize()
revision['Date_fx'] = pd.to_datetime(revision['Date_fx']).dt.normalize()

# 2. Ahora el filtro .isin() debería funcionar perfectamente
# Obtenemos los valores únicos de revisión para acelerar el proceso
fechas_a_revisar = revision['Date_fx'].unique()

# 1. Convertimos ambas columnas a datetime y eliminamos la hora (se vuelve 00:00:00)
lujan['Date_fx'] = pd.to_datetime(lujan['Date_fx']).dt.normalize()
lujan_filtrada = lujan[lujan['Date_fx'].isin(fechas_a_revisar)].copy()

print(f"Registros encontrados: {len(lujan_filtrada)}")
lujan_filtrada.head(2)

In [ ]:
lujan.columns

In [ ]:
lujan_almacen_filtro = lujan_filtrada.groupby(['Date_fx','Año_mes', 'unico_almacen', 'Sociedad', 'Almacen',
     'FechaSolicitud',  'Litros Entregado' ]).agg({ 
    'Litros': 'sum',
    'LitrosSolicitado':'last',  
    'Nro. Remito': 'last', 
    'NroPedido': 'last',
    'Fecha Remito': 'last', 
    'Usuario': 'last', 
    'Fecha Hora': 'last', 
}).reset_index()  
#lujan_almacen_filtro.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\Conciliacion\filtro_lujan.xlsx", index=False)
telemetria.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\Conciliacion\filtro_telemetria.xlsx", index=False)


# Kardex

In [ ]:
invt1 = r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\Recepciones20260508202630marzo.csv"
ingresos = pd.read_csv(invt1, sep=";", thousands=',')
ingresos.head(50)
len(ingresos)

ingresos['Fecha'] = pd.to_datetime(ingresos['Fecha'], format='%Y/%m/%d %H:%M', errors='coerce')
# 1. Asegurar orden y tipos (opcional pero recomendado)
ingresos = ingresos.sort_values(['Sitio', 'Fecha'])

In [ ]:
ingresos.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\Inventario_lujan_31_marzo.xlsx")

In [ ]:
# 2. Agrupación precisa
agrupado_1 = ingresos.groupby([pd.Grouper(key='Fecha', freq='D'), 'Sitio', 'Tanque']).agg(
    Volumen_TC_Inicial=('Volumen TC Inicial', 'first'),
    Volumen_TC_Final=('Volumen TC Final', 'last'),
    # Cambiamos el nombre para que coincida con el cálculo de abajo
    Entradas_Sonda=('Volumen Recibido', 'sum'), 
    Volumen_Agua=('Volumen de Agua Final', 'last'),
    # Cambiamos 'avg' por 'mean'
    Temperatura=('Temperatura Final', 'mean') 
)

# 3. Cálculo de Salidas
# Ahora los nombres coinciden perfectamente
agrupado_1['Salidas'] = (
    agrupado_1['Volumen_TC_Inicial'] - 
    agrupado_1['Volumen_TC_Final'] + 
    agrupado_1['Entradas_Sonda']
)

# 4. (Opcional) Resetear el índice para que sea una tabla plana
agrupado_1 = agrupado_1.reset_index()

In [ ]:
agrupado_1.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\Kardes_lujan_31_marzo.xlsx")

In [ ]:
# 1. Asegurar orden cronológico
ingresos = ingresos.sort_values(['Sitio', 'Fecha'])

# 2. Agrupación precisa
agrupado = ingresos.groupby([pd.Grouper(key='Fecha', freq='D'), 'Sitio']).agg(
    Stock_Apertura=('Volumen TC Inicial', 'first'),
    Stock_Cierre=('Volumen TC Final', 'last'),
    # Temp_Promedio=('Temperatura', 'mean'),
    # Agua_Max=('Volumen de Agua', 'max'),
    Entradas_Sonda=('Volumen Recibido', 'sum')
)

# 3. Calcular la variación física del tanque
agrupado['Variacion_Fisica'] = agrupado['Stock_Cierre'] - agrupado['Stock_Apertura']

In [ ]:
ingresos.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\marzo_vista.xlsx")

# Inventario -  Movimientos Telemetria

In [2]:
def cambio_fechas(df, fecha):
    # 1. Reemplazamos el texto exacto por 0
    df[fecha] = pd.to_datetime(df[fecha], format='%Y/%m/%d %H:%M', errors='coerce')
   
    return df

def cambio_fechas_segundos(df, fecha):
    # 1. Reemplazamos el texto exacto por 0
    df[fecha] = pd.to_datetime(df[fecha], format='%Y/%m/%d %H:%M:%S', errors='coerce')
   
    return df

In [3]:
def convertir_a_float(df, columna):
    # 1. Reemplazamos el texto exacto por 0
    df[columna] = df[columna].replace("Sin Informacion", 0)
    
    # 2. Convertimos a número. 
    # 'coerce' transforma cualquier otro texto no deseado en NaN
    df[columna] = pd.to_numeric(df[columna], errors='coerce')
    
    # 3. (Opcional) Si quieres que los errores/NaN también sean 0
    df[columna] = df[columna].fillna(0)
    
    return df

In [4]:
def limpiar_numero(valor) -> float:
    """
    Convierte strings con formato europeo a float.
    Ejemplos: "14.000,00" -> 14000.00 | "1.234" -> 1234.0
    
    Args:
        valor: String, int, float o None
    
    Returns:
        float: Número limpio o 0.0 si no se puede convertir
    """
    if pd.isna(valor) or valor == "":
        return 0.0
    
    if isinstance(valor, (int, float)):
        return float(valor)
    
    try:
        # Eliminar puntos (separador de miles) y reemplazar coma por punto decimal
        s = str(valor).strip().replace(',', '')
        return float(s)
    except (ValueError, AttributeError):
        print(f"⚠️ Advertencia: No se pudo convertir '{valor}' a número")
        return 0.0

In [5]:
def convertir_a_float(df, columna):
    # 1. Reemplazamos el texto exacto por 0
    df[columna] = df[columna].replace("Sin Informacion", 0)
    
    # 2. Convertimos a número. 
    # 'coerce' transforma cualquier otro texto no deseado en NaN
    df[columna] = pd.to_numeric(df[columna], errors='coerce')
    
    # 3. (Opcional) Si quieres que los errores/NaN también sean 0
    df[columna] = df[columna].fillna(0)
    
    return df

In [6]:
def definir_categoria(valor):
    if valor >= 1:
        return 'Entrega'  # O 'Entrega' según tu signo
    elif valor <= -1:
        return 'Recarga' # O 'Recarga' según tu signo
    else:
        # Valores entre -1 y 1 se consideran sin movimiento significativo
        return 'Sin Movimiento'


    

In [200]:
HEADERS_FIJOS_2 =  ['Fecha', 'Fecha Host', 'Fecha Actualizaci¢n', 'Sitio', 'Tanque',
       'Combustible', 'Volumen', 'Volumen TC', 'Merma', 'Volumen de Agua',
       'Temperatura', 'Altura de Combustible', 'Altura de Agua', 'Longitud',
       'Latitud', 'Altitud']

HEADERS_FINAL_2 =  ['Fecha', 'Fecha Host', 'Fecha Actualizacion', 'Sitio', 'Tanque',
       'Combustible', 'Volumen', 'Volumen TC', 'Merma', 'Volumen de Agua',
       'Temperatura', 'Altura de Combustible', 'Altura de Agua', 'Longitud',
       'Latitud', 'Altitud', 'file']

# 4. Lectura masiva archivos csv 
def read_csv_file_combustible(file_path: Path) -> pd.DataFrame | None:
    try:
        # Detectamos las columnas del archivo
        temp_df = pd.read_csv(file_path,  sep=";", thousands=';', encoding='latin1')
        cols_detectadas = HEADERS_FIJOS_2

        # Leemos el archivo completo con esas columnas corregidas
        df = pd.read_csv(file_path,  sep=";", thousands=';', encoding='latin1')

        # Reindexamos para que coincida con el orden estándar
        df = df.reindex(columns=HEADERS_FIJOS_2, fill_value=0)

        # Agregamos metadatos
        df['file'] = file_path.stem
        return df
    except Exception as e:
        print(f"Error al leer el archivo {file_path}: {e}")
        return None


# Uso
path2 = Path(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Inventario")
dfs2 = [df2 for f2 in path2.glob('*.csv') if (df2 := read_csv_file_combustible(f2)) is not None]
dfs_combustible = pd.concat(dfs2, ignore_index=True)
dfs_combustible.columns = HEADERS_FINAL_2

# Telemetria

In [22]:
# " Se convierte de Excel a CSV para eliminar caracteres invisibles"
invt1 = r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\Inventarios20260523.csv"
dfs_combustible = pd.read_csv(invt1, sep=";", thousands=';', encoding='latin1')

registros_procesados = dfs_combustible.copy()
print(R"Total Registros: {0}".format(len(registros_procesados)))

Total Registros: 40


In [23]:
registros_procesados.columns

Index(['Fecha', 'Fecha Host', 'Fecha Actualizaci¢n', 'Sitio', 'Tanque',
       'Combustible', 'Volumen', 'Volumen TC', 'Merma', 'Volumen de Agua',
       'Temperatura', 'Altura de Combustible', 'Altura de Agua', 'Longitud',
       'Latitud', 'Altitud'],
      dtype='str')

In [24]:
registros_procesados = registros_procesados.rename(columns={'Fecha Actualizaci¢n': 'Fecha Actualizacion'})
########

# Fechas
registros_procesados = cambio_fechas(registros_procesados, 'Fecha')
registros_procesados = cambio_fechas_segundos(registros_procesados, 'Fecha Host')
registros_procesados = cambio_fechas_segundos(registros_procesados, 'Fecha Actualizacion')

# Números
registros_procesados['Volumen'] = registros_procesados['Volumen'].apply(limpiar_numero)

registros_procesados['Volumen TC'] = registros_procesados['Volumen TC'].apply(limpiar_numero)
registros_procesados['Volumen de Agua'] = registros_procesados['Volumen de Agua'].apply(limpiar_numero)
registros_procesados['Merma'] = registros_procesados['Merma'].apply(limpiar_numero)
registros_procesados['Altura de Combustible'] = registros_procesados['Altura de Combustible'].apply(limpiar_numero)



In [25]:

# 1. Asegurar orden y tipos (opcional pero recomendado)
registros_procesados = registros_procesados.sort_values(['Sitio', 'Tanque', 'Fecha'])


In [26]:
registros_procesados['Variacion Volumen'] = registros_procesados['Variacion Volumen'] = (
    registros_procesados
    .groupby(['Sitio', 'Tanque'])['Volumen']
    .diff()
)


registros_procesados['Categoria'] = registros_procesados['Variacion Volumen'].apply(definir_categoria)

registros_procesados['Variacion Volumen TC'] = registros_procesados['Variacion Volumen TC'] = (
    registros_procesados
    .groupby(['Sitio', 'Tanque'])['Volumen TC']
    .diff()
)

registros_procesados['Variacion Agua'] = registros_procesados['Volumen de Agua'] = (
    registros_procesados
    .groupby(['Sitio', 'Tanque'])['Volumen de Agua']
    .diff()
)
registros_procesados['Variacion Temperat'] = registros_procesados['Temperatura'] = (
    registros_procesados
    .groupby(['Sitio', 'Tanque'])['Temperatura']
    .diff()
)

In [27]:
# 1. Calcular la diferencia de tiempo (asegurando orden por Sitio y Fecha)
registros_procesados['Fecha Actualizacion'] = pd.to_datetime(registros_procesados['Fecha Actualizacion'])
registros_procesados = registros_procesados.sort_values(['Sitio', 'Fecha Actualizacion'])
registros_procesados['Tiempo_Transcurrido'] = registros_procesados.groupby(['Sitio'])['Fecha Actualizacion'].diff()

# 2. Convertir la diferencia a HORAS (número decimal)
# Usamos total_seconds() / 3600 para obtener las horas exactas
registros_procesados['Horas_Sin_Actualizar'] = registros_procesados['Tiempo_Transcurrido'].dt.total_seconds() / 3600

# 3. Función de Alerta modificada para 1 hora
def alerta_inactividad(horas):
    if pd.isna(horas): 
        return "Inicio de Periodo"
    
    # Si pasa más de 1 hora (60 minutos)
    if horas > 1:
        # Si es más de 24 horas, lo mostramos en días para mejor lectura, si no, en horas
        if horas >= 24:
            dias = horas / 24
            return f"🚨 ALERTA: {round(dias, 1)} días sin reporte"
        else:
            return f"⚠️ ALERTA: {round(horas, 1)} horas sin reporte"
            
    return "OK (Conexión Activa)"

# 4. Aplicar la alerta
registros_procesados['Estado_Conexion'] = registros_procesados['Horas_Sin_Actualizar'].apply(alerta_inactividad)

In [28]:
# volumen > 10% y volumen < 90%

# 1. Calculamos la diferencia de tiempo (Timestamps)
# Nota: He corregido el posible error de dedo en 'Fecha Actualizaciono'
registros_procesados['Tiempo_Delay'] = registros_procesados['Fecha Actualizacion'] - registros_procesados['Fecha']

# 2. Convertimos a horas (decimal) en una columna propia
registros_procesados['Horas_Delay'] = registros_procesados['Tiempo_Delay'].dt.total_seconds() / 3600

# 3. Función de Alerta Mejorada
def alerta_delay(horas):
    if pd.isna(horas): 
        return "Sin Datos"
    
    # Error de lógica en el sistema (Relojes desincronizados)
    if horas < 0:
        return "⚠️ ERROR: Fecha Actualización anterior a Registro"
    
    # Retraso Crítico (Más de 3 horas según tu lógica)
    if horas > 3:
        return f"🚨 CRÍTICO: {round(horas, 1)} hrs de retraso"
    
    # Retraso Moderado (Opcional, por ejemplo entre 1 y 3 horas)
    if horas > 1:
        return f"🟡 ADVERTENCIA: {round(horas, 1)} hrs de retraso"
    
    return "OK: Transmisión Normal"

# 4. Aplicamos la función a una nueva columna de estado
registros_procesados['Estado_Delay'] = registros_procesados['Horas_Delay'].apply(alerta_delay)

In [29]:
registros_procesados = registros_procesados[registros_procesados['Sitio'] == "903 - Barracas Lujan"]

registros_procesados = registros_procesados[[
    'Fecha', 'Fecha Host',  'Fecha Actualizacion', 'Sitio', 'Tanque',  'Categoria',
    'Volumen','Variacion Volumen', 'Volumen TC', 'Variacion Volumen TC', 'Merma', 'Variacion Agua','Variacion Temperat',
    'Temperatura', 'Altura de Combustible', 'Altura de Agua', 
    'Tiempo_Transcurrido', 'Horas_Sin_Actualizar',
    'Estado_Conexion'
]]



registros_procesados.head(20)
registros_procesados = registros_procesados.sort_values(['Sitio', 'Fecha'])
registros_procesados.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Telemetria_procesado_completo23-mayo.xlsx", index=False)

In [18]:
telemetria_recargas = registros_procesados[registros_procesados['Categoria'] == 'Recarga']
print(R"Total Registros Recargas: {0}".format(len(telemetria_recargas)))
telemetria_recargas = telemetria_recargas.sort_values(['Sitio', 'Fecha'])
telemetria_recargas.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Telemetria_procesado_recargas_completo-0523.xlsx", index=False)

Total Registros Recargas: 22


In [ ]:


telemetria_recargas['fx_almace'] = '903 BARRACAS Lujan'
telemetria_recargas['Date_fx'] = pd.to_datetime(telemetria_recargas['Fecha'], format='%Y/%m/%d %H:%M:%S', errors='coerce').dt.date
telemetria_recargas['Año_mes'] = pd.to_datetime(telemetria_recargas['Date_fx']).dt.to_period('M').astype(str)

# 1. Asegurar que la columna sea datetime y esté ORDENADA (crítico para que funcione)
telemetria_recargas['Fecha'] = pd.to_datetime(telemetria_recargas['Fecha'])


telemetria_recargas = telemetria_recargas.sort_values('Fecha')

# 2. Calcular la diferencia de tiempo con la fila anterior (en minutos)
# .diff() resta la fila actual menos la anterior
telemetria_recargas['minutos_desde_ultimo'] = telemetria_recargas['Fecha'].diff().dt.total_seconds() / 40

# 3. Identificar saltos mayores a 60 minutos (esto define el inicio de un nuevo bloque)
# Si es la primera fila (NaN) o el tiempo es > 60, marcamos como True (1)
umbral = 60
telemetria_recargas['es_nuevo_bloque'] = (telemetria_recargas['minutos_desde_ultimo'] > umbral) | (telemetria_recargas['minutos_desde_ultimo'].isna())

# 4. CREAR EL ID DE BLOQUE
# .cumsum() sumará 1 cada vez que encuentre un 'True', creando grupos únicos: 1, 1, 1, 2, 2, 3...
telemetria_recargas['bloque_id'] = telemetria_recargas['es_nuevo_bloque'].cumsum()

# 5. Sumarizar los litros (Ingresos) por ese nuevo ID
resumen_telemetria_recargas = telemetria_recargas.groupby(['bloque_id', 'Tanque']).agg({
    'Fecha': ['max'], # Para saber cuándo empezó y terminó la descarga
    'Variacion Volumen TC': 'sum',                 # Sumamos los litros de ese bloque
    'es_nuevo_bloque': 'count', 
    #'unico_almacen' :'last', 
    'Date_fx' :'last', 
    'fx_almace':'last',                # Para saber cuántos registros enviaron la sonda
}).reset_index()

# Limpiar los nombres de las columnas para que sea fácil leer
resumen_telemetria_recargas.columns = ['ID_Bloque','Tanque',  'Fecha_Fin', 'Litros_Telemetria', 'Cant_Registros', 'Date_fx', 'fx_almace', ]
# resumen_telemetria_recargas['Date_fx'] = pd.to_datetime(resumen_telemetria_recargas['Date_fx'])
resumen_telemetria_recargas['unico_almacen'] = resumen_telemetria_recargas['fx_almace'] + '_' + resumen_telemetria_recargas['Tanque'].astype(str) + '_' + resumen_telemetria_recargas['Date_fx'].astype(str) + '_' + resumen_telemetria_recargas['Litros_Telemetria'].astype(str)

resumen_telemetria_recargas.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Telemetria-Agrupado-Recargas.xlsx", index=False)

In [220]:

telemetria_recargas['fx_almace'] = '903 BARRACAS Lujan'
telemetria_recargas['Date_fx'] = pd.to_datetime(telemetria_recargas['Fecha'], format='%Y/%m/%d %H:%M:%S', errors='coerce').dt.date
telemetria_recargas['Año_mes'] = pd.to_datetime(telemetria_recargas['Date_fx']).dt.to_period('M').astype(str)

In [224]:
# 5. Sumarizar los litros (Ingresos) por ese nuevo ID
resumen_telemetria_recargas_agrupada = telemetria_recargas.groupby(['Date_fx','Tanque' ]).agg({
    'Variacion Volumen TC': 'sum',                 # Sumamos los litros de ese bloque
    'Fecha': 'count', 
    'fx_almace':'last',                # Para saber cuántos registros enviaron la sonda
}).reset_index()

# Limpiar los nombres de las columnas para que sea fácil leer
resumen_telemetria_recargas_agrupada.columns = ['Date_fx', 'Tanque', 'Litros_Telemetria', 'Cant_Registros', 'fx_almace' ]
# resumen_telemetria_recargas['Date_fx'] = pd.to_datetime(resumen_telemetria_recargas['Date_fx'])
resumen_telemetria_recargas_agrupada['unico_almacen'] = resumen_telemetria_recargas_agrupada['fx_almace'] + '_' + resumen_telemetria_recargas_agrupada['Tanque'].astype(str) + '_' + resumen_telemetria_recargas_agrupada['Date_fx'].astype(str) + '_' + resumen_telemetria_recargas_agrupada['Litros_Telemetria'].astype(str)
resumen_telemetria_recargas = resumen_telemetria_recargas_agrupada.copy()
resumen_telemetria_recargas.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Telemetria-Agrupado-Recargas-V2.xlsx", index=False)

## Entregas Telemetria

In [31]:
telemetria = registros_procesados[registros_procesados['Categoria'] == 'Entrega']
print(R"Total Registros Entregas: {0}".format(len(telemetria)))
telemetria = telemetria.sort_values(['Sitio', 'Fecha'])
telemetria.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Telemetria_procesado_entregas_mayo-23.xlsx", index=False)

Total Registros Entregas: 4


In [20]:


telemetria['fx_almace'] = '903 BARRACAS Lujan'
telemetria['Date_fx'] = pd.to_datetime(telemetria['Fecha'], format='%Y/%m/%d %H:%M:%S', errors='coerce').dt.date
telemetria['Año_mes'] = pd.to_datetime(telemetria['Date_fx']).dt.to_period('M').astype(str)

# 1. Asegurar que la columna sea datetime y esté ORDENADA (crítico para que funcione)
telemetria['Fecha'] = pd.to_datetime(telemetria['Fecha'])


telemetria = telemetria.sort_values('Fecha')

# 2. Calcular la diferencia de tiempo con la fila anterior (en minutos)
# .diff() resta la fila actual menos la anterior
telemetria['minutos_desde_ultimo'] = telemetria['Fecha'].diff().dt.total_seconds() / 40

# 3. Identificar saltos mayores a 60 minutos (esto define el inicio de un nuevo bloque)
# Si es la primera fila (NaN) o el tiempo es > 60, marcamos como True (1)
umbral = 60
telemetria['es_nuevo_bloque'] = (telemetria['minutos_desde_ultimo'] > umbral) | (telemetria['minutos_desde_ultimo'].isna())

# 4. CREAR EL ID DE BLOQUE
# .cumsum() sumará 1 cada vez que encuentre un 'True', creando grupos únicos: 1, 1, 1, 2, 2, 3...
telemetria['bloque_id'] = telemetria['es_nuevo_bloque'].cumsum()

# 5. Sumarizar los litros (Ingresos) por ese nuevo ID
resumen_telemetria = telemetria.groupby('bloque_id').agg({
    'Fecha': ['min', 'max'], # Para saber cuándo empezó y terminó la descarga
    'Variacion Volumen TC': 'sum',                 # Sumamos los litros de ese bloque
    'es_nuevo_bloque': 'count', 
    #'unico_almacen' :'last', 
    'Date_fx' :'last', 
    'fx_almace':'last'               # Para saber cuántos registros enviaron la sonda
}).reset_index()

# Limpiar los nombres de las columnas para que sea fácil leer
resumen_telemetria.columns = ['ID_Bloque', 'Fecha_Inicio', 'Fecha_Fin', 'Litros_Telemetria', 'Cant_Registros', 'Date_fx', 'fx_almace']
# resumen_telemetria['Date_fx'] = pd.to_datetime(resumen_telemetria['Date_fx'])
resumen_telemetria['unico_almacen'] = resumen_telemetria['fx_almace'] + '_' + resumen_telemetria['Date_fx'].astype(str) + '_' + resumen_telemetria['Litros_Telemetria'].astype(str)

resumen_telemetria['Date_fx'] = pd.to_datetime(resumen_telemetria['Date_fx']).dt.normalize().astype('datetime64[ns]')
resumen_telemetria.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Telemetria-Analisis-Entregas_mayo23.xlsx", index=False)

## Conciliación

In [ ]:
resumen_telemetria = resumen_telemetria.drop_duplicates(subset=['unico_almacen'])


# 1. Ejecutar el merge_asof original
prueba_3 = pd.merge_asof(
    resumen_telemetria.sort_values("Date_fx"),
    df_combustible_lujan.sort_values("Date_fx"),
    on="Date_fx",
    by='unico_almacen',
    tolerance=pd.Timedelta(days=2),
    direction='nearest'
)

# 2. Encontrar qué registros de lujan_almacen NO están en prueba_3
# Usamos un indicador para ver qué falta
almacen_en_prueba = prueba_3[['Date_fx', 'unico_almacen', 'Litros_GB']].dropna()



######################
# Identificamos los registros de lujan_almacen que no cruzaron
perdidos = df_combustible_lujan[~df_combustible_lujan.set_index(['Date_fx', 'unico_almacen']).index.isin(
    prueba_3.set_index(['Date_fx', 'unico_almacen']).index
)].copy()

# 3. Unir los registros perdidos al dataset de conciliación
# Llenamos 'Ingresos' con 0 porque son cargas de almacén que la telemetría no vio
perdidos['Litros_Telemetria'] = 0
conciliacion_completa = pd.concat([prueba_3, perdidos], ignore_index=True).sort_values([ 'Date_fx', 'NroPedido',]) 

conciliacion_completa['Dif Solic vs GB'] = conciliacion_completa['Litros_GB'] - conciliacion_completa['L_Solicitados']

# Llenar NaNs para poder restar sin errores
conciliacion_completa['Litros_Telemetria'] = conciliacion_completa['Litros_Telemetria'].fillna(0)
conciliacion_completa['Litros_GB'] = conciliacion_completa['Litros_GB'].fillna(0)

# Calcular diferencia individual
conciliacion_completa['Diferencia_Litros'] = conciliacion_completa['Litros_Telemetria'] - conciliacion_completa['Litros_GB']

# 1. Diferencia acumulada POR ALMACÉN (Correcto)
conciliacion_completa['Diferencia_Acumulada'] = conciliacion_completa.groupby('Fecha_Inicio')['Diferencia_Litros'].cumsum()

# 2. Suma móvil POR ALMACÉN (Correcto)
conciliacion_completa['Suma_Compensatoria'] = (
    conciliacion_completa.groupby('Date_fx')['Diferencia_Litros'] # Nro. Remito   Date_fx
    .rolling(window=2)
    .sum()
    .reset_index(level=0, drop=True)
)


def clasificar_error(row):
    if abs(row['Diferencia_Litros']) <= 50: # Tolerancia técnica normal
        return 'OK - Conciliado'
    elif abs(row['Suma_Compensatoria']) <= 50:
        return 'Compensado (Desfase de tiempo)'
    elif row['Diferencia_Litros'] > 300:
        return 'Sobrante (Posible carga no registrada)'
    elif row['Diferencia_Litros'] < -300:
        return 'Faltante (Posible merma o robo)'
    else:
        return 'Revisar manualmente'

conciliacion_completa['Estado_Critico'] = conciliacion_completa.apply(clasificar_error, axis=1)

def clasificar_conciliacion(row):
    # Usamos el valor absoluto para las validaciones de cercanía a cero (OK y Compensados)
    diferencia = row['Suma_Compensatoria']
    abs_diff = abs(diferencia)

    # 1. Rango de Tolerancia Técnica (Diferencias mínimas)
    if abs_diff <= 50:
        return 'OK - Conciliado'
    
    # 2. Rango de Compensación (Desfases leves de tiempo o medición)
    if abs_diff <= 100:
        return 'Compensado (Desfase de tiempo)'

    # 3. Clasificación de SOBRANTES (Valores Positivos)
    if diferencia > 100:
        if diferencia < 1000:
            return 'Sobrante entre 101 y 1000L'
        else:
            return 'Sobrante (Carga no registrada) >= 1000L'

    # 4. Clasificación de FALTANTES (Valores Negativos)
    if diferencia < -100:
        if diferencia > -1000:
            return 'Faltante (Posible merma o robo) entre -101 y -1000L'
        else:
            return 'Faltante (Crítico - Posible robo) <= -1000L'

    # 5. Caso por defecto (Seguridad)
    return 'Revisar manualmente'

conciliacion_completa['Estado_Conciliado'] = conciliacion_completa.apply(clasificar_conciliacion, axis=1)


#conciliacion_completa[(conciliacion_completa["Date_fx"] > '2025-11-01') & (conciliacion_completa["Fecha_Inicio"] < '2026-02-02')]

conciliacion_completa.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Outs\Conciliacion-Final.xlsx", index=False)


# Conciliacion Recargas

[Saltar al procesamiento Gestion Bus Recargas](#seccion_clave_unica)

In [249]:
resumen_telemetria_recargas = cambio_fechas(resumen_telemetria_recargas, 'Date_fx')
df_combustible_recargas_agrupado = cambio_fechas(df_combustible_recargas_agrupado, 'Date_fx')

In [250]:
df_combustible_recargas_agrupado = df_combustible_recargas_agrupado[['Date_fx', 'Almacen', 'Tanque',
       'Litros_GB_Recargas', 'Cantidad_veces', 'unico_almacen']]

In [251]:
df_combustible_recargas_agrupado.head(2)

,Date_fx,Almacen,Tanque,Litros_GB_Recargas,Cantidad_veces,unico_almacen
0,2025-11-03,903 BARRACAS Lujan,1,-635.0,3,903 BARRACAS Lujan_1_2025-11-03_-635.0
1,2025-11-04,903 BARRACAS Lujan,1,-15247.0,52,903 BARRACAS Lujan_1_2025-11-04_-15247.0


In [252]:
#resumen_telemetria_recargas = resumen_telemetria_recargas.drop_duplicates(subset=['unico_almacen'])


# 1. Ejecutar el merge_asof original
agrupacion_recargas = pd.merge_asof(
    resumen_telemetria_recargas.sort_values("Date_fx"),
    df_combustible_recargas_agrupado.sort_values("Date_fx"),
    on="Date_fx",
    by='unico_almacen',
    tolerance=pd.Timedelta(days=2),
    direction='nearest'
)

# 2. Encontrar qué registros de lujan_almacen NO están en prueba_recargas
# Usamos un indicador para ver qué falta
almacen_en_prueba_recargas = prueba_recargas[['Date_fx', 'unico_almacen', 'Litros_GB_Recargas']].dropna()



######################
# Identificamos los registros de lujan_almacen que no cruzaron
perdidos_recargas = df_combustible_recargas_agrupado[~df_combustible_recargas_agrupado.set_index(['Date_fx', 'unico_almacen']).index.isin(
    agrupacion_recargas.set_index(['Date_fx', 'unico_almacen']).index
)].copy()

# 3. Unir los registros perdidos al dataset de conciliación
# Llenamos 'Ingresos' con 0 porque son cargas de almacén que la telemetría no vio
perdidos_recargas['Litros_Telemetria'] = 0
conciliacion_completa_recargas = pd.concat([agrupacion_recargas, perdidos_recargas], ignore_index=True).sort_values(['Date_fx', 'Tanque']) 

# Llenar NaNs para poder restar sin errores
conciliacion_completa_recargas['Litros_Telemetria'] = conciliacion_completa_recargas['Litros_Telemetria'].fillna(0)
conciliacion_completa_recargas['Litros_GB_Recargas'] = conciliacion_completa_recargas['Litros_GB_Recargas'].fillna(0)

# Calcular diferencia individual
conciliacion_completa_recargas['Diferencia_Litros'] = conciliacion_completa_recargas['Litros_Telemetria'] - conciliacion_completa_recargas['Litros_GB_Recargas']

# 1. Diferencia acumulada POR ALMACÉN (Correcto)
conciliacion_completa_recargas['Diferencia_Acumulada'] = conciliacion_completa_recargas.groupby('Date_fx')['Diferencia_Litros'].cumsum()

# 2. Suma móvil POR ALMACÉN (Correcto)
conciliacion_completa_recargas['Suma_Compensatoria'] = (
    conciliacion_completa_recargas.groupby('Date_fx')['Diferencia_Litros'] # Nro. Remito   Date_fx
    .rolling(window=2)
    .sum()
    .reset_index(level=0, drop=True)
)


# def clasificar_error(row):
#     if abs(row['Diferencia_Litros']) <= 50: # Tolerancia técnica normal
#         return 'OK - Conciliado'
#     elif abs(row['Suma_Compensatoria']) <= 50:
#         return 'Compensado (Desfase de tiempo)'
#     elif row['Diferencia_Litros'] > 300:
#         return 'Sobrante (Posible carga no registrada)'
#     elif row['Diferencia_Litros'] < -300:
#         return 'Faltante (Posible merma o robo)'
#     else:
#         return 'Revisar manualmente'

# conciliacion_completa_recargas['Estado_Critico'] = conciliacion_completa_recargas.apply(clasificar_error, axis=1)

# def clasificar_conciliacion(row):
#     # Usamos el valor absoluto para las validaciones de cercanía a cero (OK y Compensados)
#     diferencia = row['Suma_Compensatoria']
#     abs_diff = abs(diferencia)

#     # 1. Rango de Tolerancia Técnica (Diferencias mínimas)
#     if abs_diff <= 50:
#         return 'OK - Conciliado'
    
#     # 2. Rango de Compensación (Desfases leves de tiempo o medición)
#     if abs_diff <= 100:
#         return 'Compensado (Desfase de tiempo)'

#     # 3. Clasificación de SOBRANTES (Valores Positivos)
#     if diferencia > 100:
#         if diferencia < 1000:
#             return 'Sobrante entre 101 y 1000L'
#         else:
#             return 'Sobrante (Carga no registrada) >= 1000L'

#     # 4. Clasificación de FALTANTES (Valores Negativos)
#     if diferencia < -100:
#         if diferencia > -1000:
#             return 'Faltante (Posible merma o robo) entre -101 y -1000L'
#         else:
#             return 'Faltante (Crítico - Posible robo) <= -1000L'

#     # 5. Caso por defecto (Seguridad)
#     return 'Revisar manualmente'

#conciliacion_completa_recargas['Estado_Conciliado'] = conciliacion_completa_recargas.apply(clasificar_conciliacion, axis=1)


#conciliacion_completa_recargas[(conciliacion_completa_recargas["Date_fx"] > '2025-11-01') & (conciliacion_completa_recargas["Fecha_Inicio"] < '2026-02-02')]

conciliacion_completa_recargas.to_excel(r"C:\Users\jpineda\Desktop\Tableros Grupo\1_OPERACIONES\2_Reporte Combustible\20_Compensacion\Telemetria\Conciliacion Recargas\Conciliacion-Recargas.xlsx", index=False)


In [248]:
conciliacion_completa_recargas.head(2)

,Date_fx,Tanque_x,Litros_Telemetria,Cant_Registros,fx_almace,unico_almacen,Almacen,Tanque_y,Litros_GB_Recargas,Cantidad_veces,Tanque,Diferencia_Litros,Diferencia_Acumulada,Suma_Compensatoria
0,2025-10-30,5.0,-401.21,2.0,903 BARRACAS Lujan,903 BARRACAS Lujan_5_2025-10-30_-401.209999999...,NaN,NaN,0.0,NaN,NaN,-401.21,-401.21,NaN
1,2025-10-30,6.0,-386.07,2.0,903 BARRACAS Lujan,903 BARRACAS Lujan_6_2025-10-30_-386.069999999...,NaN,NaN,0.0,NaN,NaN,-386.07,-787.28,-787.28


In [ ]:
conciliacion_completa_recargas[['Tanque_x', 'Fecha_Fin', 'Litros_Telemetria',
       'Cant_Registros', 'Date_fx', 'fx_almace', 'unico_almacen', 'Año_mes',
       'unico_mes',  , 'Litros_GB_Recargas',
       'Cantidad_veces', 'Surtidor', 'Tanque', 'Diferencia_Litros',
       'Diferencia_Acumulada', 'Suma_Compensatoria', 'Estado_Critico']]
